Code base sourced from
1. Andrej Karpathy [GPT From Scratch](https://colab.research.google.com/drive/1yd0yFHmw63G8xq3EBN_EN1C6Rdiv4GSS?usp=sharing#scrollTo=wJpXpmjEYC_T)
2. [Wikipedia Text Generation Notebook](https://colab.research.google.com/github/trekhleb/machine-learning-experiments/blob/master/experiments/text_generation_wikipedia_rnn/text_generation_wikipedia_rnn.ipynb#scrollTo=6dj4e-AGMaV4)

In [ ]:
!pip install torch datasets tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import os
import time

# Hyperparameters

In [ ]:
batch_size = 32  # sequences to process in parallel
block_size = 128  # maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 256
n_head = 8
n_layer = 6
dropout = 0.1

In [ ]:
# Data parameters
num_articles = 50000 #10000  # Number of articles to use (set to None for full dataset)
min_article_length = 500  # Minimum characters in article

In [ ]:
print(f"Using device: {device}")
torch.manual_seed(42)

# Transformer Components

In [ ]:

class Head(nn.Module):
    """One head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # (B, T, T)
        wei = F.softmax(wei, dim=-1)  # (B, T, T)
        wei = self.dropout(wei)

        # Perform weighted aggregation of values
        v = self.value(x)  # (B,T,head_size)
        out = wei @ v  # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """A simple linear layer followed by a non-linearity"""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: communication followed by computation"""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class TransformerLanguageModel(nn.Module):
    """Transformer-based language model"""

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)  # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T,C)
        x = tok_emb + pos_emb  # (B,T,C)
        x = self.blocks(x)  # (B,T,C)
        x = self.ln_f(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate new tokens"""
        for _ in range(max_new_tokens):
            # Crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # Get the predictions
            logits, loss = self(idx_cond)
            # Focus only on the last time step
            logits = logits[:, -1, :] / temperature  # becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

# Data Loading and Processing

In [ ]:
def load_wikipedia_data():
    """Load Wikipedia dataset from HuggingFace"""
    print("Loading Wikipedia dataset...")
    ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

    # Collect articles
    texts = []
    print(f"Collecting {num_articles if num_articles else 'all'} articles...")

    for i, example in enumerate(tqdm(ds, total=num_articles)):
        if num_articles and i >= num_articles:
            break

        text = example['text']
        # Filter out very short articles
        if len(text) >= min_article_length:
            texts.append(text)

    print(f"Collected {len(texts)} articles")

    # Combine all texts
    combined_text = '\n\n'.join(texts)
    return combined_text


def create_vocab_and_encode(text):
    """Create vocabulary and encoding/decoding functions"""
    # Get all unique characters
    chars = sorted(list(set(text)))
    vocab_size = len(chars)

    print(f"Vocabulary size: {vocab_size} unique characters")

    # Create character mappings
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    # Encoder: string to list of integers
    encode = lambda s: [stoi[c] for c in s]
    # Decoder: list of integers to string
    decode = lambda l: ''.join([itos[i] for i in l])

    return vocab_size, encode, decode, stoi, itos


def prepare_data(text, encode):
    """Prepare train/val/test splits"""
    print("Encoding text...")
    data = torch.tensor(encode(text), dtype=torch.long)

    # Split into train/val/test
    n = len(data)
    train_size = int(0.8 * n)
    val_size = int(0.1 * n)

    train_data = data[:train_size]
    val_data = data[train_size:train_size + val_size]
    test_data = data[train_size + val_size:]

    print(f"Train size: {len(train_data):,} characters")
    print(f"Val size: {len(val_data):,} characters")
    print(f"Test size: {len(test_data):,} characters")

    return train_data, val_data, test_data

# Training Functions

In [ ]:
def get_batch(data, batch_size, block_size):
    """Generate a small batch of data"""
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model, train_data, val_data):
    """Estimate loss on train and val sets"""
    out = {}
    model.eval()

    for split, data in [('train', train_data), ('val', val_data)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(data, batch_size, block_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out


def train_model(model, train_data, val_data, optimizer):
    """Training loop"""
    print("\n" + "="*50)
    print("Starting training...")
    print("="*50 + "\n")

    best_val_loss = float('inf')

    for iter in range(max_iters):
        # Evaluate loss periodically
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss(model, train_data, val_data)
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

            # Save best model
            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                torch.save({
                    'iter': iter,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': losses['val'],
                }, 'best_model.pt')
                print(f"  → Saved new best model (val loss: {best_val_loss:.4f})")

        # Sample a batch of data
        xb, yb = get_batch(train_data, batch_size, block_size)

        # Evaluate the loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    print("\n" + "="*50)
    print("Training completed!")
    print("="*50 + "\n")


def generate_text(model, decode, start_string, max_new_tokens=500, temperature=0.8):
    """Generate text from the model"""
    model.eval()

    # Encode the starting string
    context = torch.tensor([encode(start_string)], dtype=torch.long, device=device)

    # Generate
    generated = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature)

    # Decode and return
    return decode(generated[0].tolist())

# Main Execution

In [ ]:
def main():
    global encode, decode  # Make these available for generate_text

    print("\n" + "="*70)
    print("WIKIPEDIA TEXT GENERATION WITH TRANSFORMER")
    print("="*70 + "\n")

    # Step 1: Load data
    text = load_wikipedia_data()

    # Step 2: Create vocabulary
    vocab_size, encode, decode, stoi, itos = create_vocab_and_encode(text)

    # Step 3: Prepare data splits
    train_data, val_data, test_data = prepare_data(text, encode)

    # Step 4: Initialize model
    print("\nInitializing model...")
    model = TransformerLanguageModel(vocab_size)
    model = model.to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model has {num_params/1e6:.2f}M parameters")

    # Step 5: Create optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # Step 6: Train model
    train_model(model, train_data, val_data, optimizer)

    # Step 7: Load best model
    print("\nLoading best model for generation...")
    checkpoint = torch.load('best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model from iteration {checkpoint['iter']} with val loss {checkpoint['val_loss']:.4f}")

    # Step 8: Generate samples
    print("\n" + "="*70)
    print("GENERATED SAMPLES")
    print("="*70 + "\n")

    start_strings = [
        "Science is",
        "The history of",
        "In mathematics,",
        "The theory of"
    ]

    temperatures = [0.5, 0.8, 1.0]

    for start_string in start_strings:
        print(f"\n{'='*70}")
        print(f"Starting with: '{start_string}'")
        print('='*70)

        for temp in temperatures:
            print(f"\n--- Temperature: {temp} ---")
            generated = generate_text(model, decode, start_string,
                                     max_new_tokens=300, temperature=temp)
            print(generated)
            print()

    # Step 9: Evaluate on test set
    print("\n" + "="*70)
    print("FINAL TEST SET EVALUATION")
    print("="*70 + "\n")

    model.eval()
    test_losses = torch.zeros(eval_iters)

    for k in range(eval_iters):
        X, Y = get_batch(test_data, batch_size, block_size)
        logits, loss = model(X, Y)
        test_losses[k] = loss.item()

    test_loss = test_losses.mean()
    print(f"Test loss: {test_loss:.4f}")

    # Save final model
    torch.save({
        'model_state_dict': model.state_dict(),
        'vocab_size': vocab_size,
        'stoi': stoi,
        'itos': itos,
        'hyperparameters': {
            'n_embd': n_embd,
            'n_head': n_head,
            'n_layer': n_layer,
            'block_size': block_size,
            'dropout': dropout
        }
    }, 'final_model.pt')

    print("\nFinal model saved to 'final_model.pt'")
    print("\n" + "="*70)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*70 + "\n")

    return model, encode, decode


if __name__ == "__main__":
    model, encode, decode = main()